## Initiate Gemini API

In [12]:
"""
Рабочий пример с официальным SDK google‑genai.
Вариант с загрузкой API‑ключа из .env файла.
Установите пакет: pip install --upgrade google-genai python-dotenv
"""
from google import genai
import os
from dotenv import load_dotenv

# Загружаем переменные окружения из .env файла
load_dotenv()

# Получаем API ключ из переменной окружения
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

In [13]:
# Available models
print([model.name for model in gemini_client.models.list()])

['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-exp-1206', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-preview-09-2025', 'models/gemini-2.5-flash-lite-preview-09-2025', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'models/gemini-robotics-er-1.5-preview', 'models/gemini-2.5-computer-use-preview-10-2025', 'models/deep-research-pro-preview-12-2025', 'models/gemini-embedding-001', 'models/aqa', 'mode

In [14]:
resp = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[{"role": "user", "parts": [{"text": "Напиши короткое стихотворение про Python"}]}],
)

print(resp.text)

Питон, ты словно змейка в танце,
Твой код читаем, без нюансов.
От скриптов простых до больших систем,
Решаешь тысячи проблем.

Библиотек твоих богатство,
Откроет кода целое царство.
Для веб, AI, машин, наук –
Ты верный друг, без лишних мук!


## Initiate OpenAI compatible Gemini client

In [14]:
from openai import OpenAI

gemini_client_openai_compatible = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

## Initiate LangChain with local model using ollama

In [21]:
from langchain_ollama import ChatOllama

# Initialize the Ollama model
langchain_ollama_llm = ChatOllama(
    model="qwen3:8b",
    base_url=os.getenv('OLLAMA_HOST'),
    temperature=0.7,
)

## Initiate LangChain with Gemini model

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize model
langchain_gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

## Prompt techniques - Few-shot prompt

In [43]:
# Few-shot prompt example with Ollama using messages format
# This demonstrates how to teach the model a pattern through examples

# Test input
user_text = "The weather today is beautiful and I'm feeling great!"

# Create messages with system, user, and assistant roles
messages = [
    {"role": "system", "content": "You are a helpful assistant that classifies text sentiment."},

    # Example 1
    {"role": "user", "content": "I love this product! It exceeded my expectations."},
    {"role": "assistant", "content": "Positive"},

    # Example 2
    {"role": "user", "content": "The service was terrible and the staff was rude."},
    {"role": "assistant", "content": "Negative"},

    # Example 3
    {"role": "user", "content": "It's okay, nothing special but not bad either."},
    {"role": "assistant", "content": "Neutral"},

    # Example 4
    {"role": "user", "content": "This is the worst experience I've ever had!"},
    {"role": "assistant", "content": "Negative"},

    # Example 5
    {"role": "user", "content": "Absolutely amazing! Highly recommend to everyone."},
    {"role": "assistant", "content": "Positive"},

    # Actual query
    {"role": "user", "content": user_text}
]

response = langchain_ollama_llm.invoke(messages)
print(f"Input: {user_text}")
print(f"Classification: {response.content}")

Input: The weather today is beautiful and I'm feeling great!
Classification: Positive


## Single agent

In [28]:
# Agent with tools example using LangChain and Ollama
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.messages import SystemMessage, HumanMessage

# Define two tools for the agent to use
@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression. Use this for calculations.

    Args:
        expression: A mathematical expression like '2 + 2' or '10 * 5'
    """
    try:
        result = eval(expression)
        return f"The result is: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def text_analyzer(text: str) -> str:
    """Analyzes text and returns statistics about it.

    Args:
        text: The text to analyze
    """
    word_count = len(text.split())
    char_count = len(text)
    return f"Text stats - Words: {word_count}, Characters: {char_count}"

# Create list of tools
tools = [calculate, text_analyzer]

# Create the agent using create_react_agent
agent = create_agent(model=langchain_ollama_llm, tools=tools, system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant helping with math and text analysis.",
            }
        ]
    )
)

result = agent.invoke(
    {"messages": [HumanMessage("Get text statistic for following: No, no te puedo olvidar. No, no te puedo borrar. Tú me enseñaste a querer. Me enseñaste a bailar.")]}
)
result

ConnectError: [Errno 61] Connection refused

In [40]:
print(len(result['messages']))
print(result['messages'][1].tool_calls)
print(result['messages'][2])
print(result['messages'][3].content)

4
[{'name': 'text_analyzer', 'args': {'text': 'No, no te puedo olvidar. No, no te puedo borrar. Tú me enseñaste a querer. Me enseñaste a bailar.'}, 'id': '27a1542d-18a0-4216-b62c-0c197a7f3899', 'type': 'tool_call'}]
content='Text stats - Words: 19, Characters: 97' name='text_analyzer' id='3ce7e722-fb86-406e-88f4-d6129ab1a0c1' tool_call_id='27a1542d-18a0-4216-b62c-0c197a7f3899'
The text statistics are as follows:  
- **Words**: 19  
- **Characters**: 97  

Let me know if you need further analysis!


## Multi-Agent system

In [45]:
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

@tool
def calculate(expression: str) -> float | str:
    """Безопасно вычисляет математическое выражение"""
    try:
        result = eval(expression)
        return f"The result is: {result}"
    except Exception as e:
        return "no result"

@tool
def convert_currency(amount: float, from_cur: str, to_cur: str) -> float | str:
    """Конвертирует валюту. Курсы: USD/KZT=480, EUR/KZT=520, USD/EUR=1.1

    Args:
        amount: Сумма
        from_cur: Исходная валюта (USD/EUR/KZT)
        to_cur: Целевая валюта

    Returns:
        Сконвертированная сумма
    """
    rates = {
        'USD/KZT': 495.451692,
        'KZT/USD': 0.002018,
        'EUR/KZT': 588.295373,
        'KZT/Euro':	0.001700,
        'GBP/KZT': 675.394799,
        'KZT/GBP': 0.001481
    }
    conversion_rate = rates.get(f"{from_cur}/{to_cur}", 0)
    return amount * conversion_rate if conversion_rate > 0 else 'Not found information'

@tool
def get_weather(city: str) -> str:
    """Получает погоду для города"""
    weather_db = {
        "Астана": "Солнечно, -5°C",
        "Алматы": "Облачно, +2°C",
        "Шымкент": "Ясно, +8°C"
    }
    return weather_db.get(city, "Couldn't find information")

weather_agent = AgentExecutor(
    agent=create_tool_calling_agent(
        langchain_ollama_llm,
        [get_weather],
        ChatPromptTemplate.from_messages([
            ("system", "Ты эксперт по погоде."),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}")
        ])
    ),
    tools=[get_weather],
    name="weather_agent"
)

finance_agent = AgentExecutor(
    agent=create_tool_calling_agent(
        langchain_ollama_llm,
        [convert_currency, calculate],
        ChatPromptTemplate.from_messages([
            ("system", "Ты эксперт ыполняющий финансовые расчеты и конвертацию валют."),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}")
        ])
    ),
    tools=[convert_currency, calculate],
    name="finance_agent"
)

@tool
def weather_agent_as_tool(query: str) -> str:
    """Получает информацию о погоде"""
    return weather_agent.invoke({"input": query})["output"]

@tool
def finance_agent_as_tool(query: str) -> str:
    """Выполняет финансовые расчеты и конвертацию"""
    return finance_agent.invoke({"input": query})["output"]

orchestrator = AgentExecutor(
    agent=create_tool_calling_agent(
        langchain_ollama_llm,
        [weather_agent_as_tool, finance_agent_as_tool],
        ChatPromptTemplate.from_messages([
            ("system", "Ты главный агента отвечающие на вопросы использую свои инструменты"),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}")
        ])
    ),
    tools=[weather_agent_as_tool, finance_agent_as_tool],
    name="orchestrator"
)

# Тест:
result = orchestrator.invoke({
    "input": "Какая погода в Астане? Если я возьму 100 USD, сколько это в тенге?"
})
result

In [47]:
result = finance_agent.invoke({
    "input": "Если я возьму 100 USD, сколько это в тенге?"
})
result

{'input': 'Если я возьму 100 USD, сколько это в тенге?',
 'output': '100 USD равно 49\u202f545,17 тенге (KZT) на основе текущего курса. Обратите внимание, что обменные курсы могут колебаться, поэтому актуальная сумма может немного отличаться.'}

## Wikipedia as tool

In [1]:
import wikipedia

# 1. Поиск статей
print("=" * 60)
print("ПОИСК СТАТЕЙ: wikipedia.search()")
print("=" * 60)
results = wikipedia.search("Python programming", results=5)
print(f"Найденные статьи: {results}")

# 2. Краткое содержание
print("\n" + "=" * 60)
print("КРАТКОЕ СОДЕРЖАНИЕ: wikipedia.summary()")
print("=" * 60)
summary = wikipedia.summary("Python (programming language)", sentences=2)
print(summary)

# 3. Полная страница
print("\n" + "=" * 60)
print("ПОЛНАЯ СТРАНИЦА: wikipedia.page()")
print("=" * 60)
page = wikipedia.page("Artificial intelligence")
print(f"Заголовок: {page.title}")
print(f"URL: {page.url}")
print(f"Количество ссылок: {len(page.links)}")
print(f"Первые 200 символов:\n{page.content[:200]}...")

ПОИСК СТАТЕЙ: wikipedia.search()
Найденные статьи: ['Python (programming language)', 'History of Python', 'Outline of the Python programming language', 'Python syntax and semantics', 'Mojo (programming language)']

КРАТКОЕ СОДЕРЖАНИЕ: wikipedia.summary()
Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation.

ПОЛНАЯ СТРАНИЦА: wikipedia.page()
Заголовок: Artificial intelligence
URL: https://en.wikipedia.org/wiki/Artificial_intelligence
Количество ссылок: 1805
Первые 200 символов:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec...


In [4]:
from langchain_core.tools import tool

@tool
def search_wikipedia(query: str) -> str:
    """
    Поиск статей в Wikipedia по запросу.

    Args:
        query: Поисковый запрос

    Returns:
        str: Список найденных статей
    """
    try:
        return wikipedia.search(query, results=5)
    except Exception as e:
        print("Got error", e)



# Тест вашей функции
print("Тест search_wikipedia:")
print(search_wikipedia.invoke({"query": "Machine Learning"}))

Тест search_wikipedia:
['Machine learning', 'Attention (machine learning)', 'Neural network (machine learning)', 'Quantum machine learning', 'Deep learning']


In [11]:
@tool
def get_summary(topic: str, sentences: int = 3) -> str:
    """
    Получить краткое содержание статьи из Wikipedia.

    Args:
        topic: Название статьи
        sentences: Количество предложений (по умолчанию 3)

    Returns:
        str: Краткое содержание статьи
    """
    try:
        return wikipedia.summary(topic, sentences=sentences)
    except wikipedia.exceptions.DisambiguationError as e:
        return "please specify your topic"
    except wikipedia.exceptions.PageError as e:
        return "no page found"
    except Exception as e:
        print("Got error", e)


# Тест вашей функции
print("Тест get_summary:")
print(get_summary.invoke({"topic": "Python (programming language)", "sentences": 2}))
print(get_summary.invoke({"topic": "Python", "sentences": 2}))

Тест get_summary:
Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation.
please specify your topic


/Users/sstamkulov/Documents/ml/jupyter_notebooks_archive/nfactorial_llm_course/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/sstamkulov/Documents/ml/jupyter_notebooks_archive/nfactorial_llm_course/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [37]:
# Привязка инструментов к модели
model_with_tools = langchain_ollama_llm.bind_tools([search_wikipedia, get_summary])

In [44]:
from langchain.messages import ToolMessage, HumanMessage, AIMessage

# Вопрос пользователя
user_question = "Give information about PlayStation Portable"

print(f"Вопрос: {user_question}\n")
print("=" * 80)

messages = [HumanMessage(user_question)]

ai_response = model_with_tools.invoke(messages)

while len(ai_response.tool_calls) > 0:
    if ai_response.content != "":
        ai_message = AIMessage(ai_response.content)
        messages.add(ai_message)

    print(f"Количество вызовов инструментов: {len(ai_response.tool_calls)}")

    for tool_call in ai_response.tool_calls:
        print(f"\nВызов инструмента: {tool_call['name']}")
        print(f"Аргументы: {tool_call['args']}")

        if tool_call['name'] == 'search_wikipedia':
            result = search_wikipedia.invoke(tool_call)
        elif tool_call['name'] == 'get_summary':
            result = get_summary.invoke(tool_call)

        tool_message = ToolMessage(
            content=result,
            tool_call_id=tool_call['id']
        )
        messages.append(tool_message)

    print("\n" + "=" * 80)
    print(messages)

    ai_response = model_with_tools.invoke(messages)

print("\n" + "=" * 80)
print("ФИНАЛЬНЫЙ ОТВЕТ:")
print("=" * 80)
print(ai_response.content)

Вопрос: Give information about PlayStation Portable

Количество вызовов инструментов: 1

Вызов инструмента: search_wikipedia
Аргументы: {'query': 'PlayStation Portable'}

[HumanMessage(content='Give information about PlayStation Portable', additional_kwargs={}, response_metadata={}), ToolMessage(content="content=['PlayStation Portable', 'List of PlayStation Portable games', 'PlayStation Portable hardware', 'PlayStation Vita', 'PlayStation Portable homebrew'] name='search_wikipedia' tool_call_id='dcd5b9ba-538f-4c27-8686-f97674231adb'", tool_call_id='dcd5b9ba-538f-4c27-8686-f97674231adb')]
Количество вызовов инструментов: 2

Вызов инструмента: get_summary
Аргументы: {'topic': 'PlayStation Portable', 'sentences': 3}

Вызов инструмента: get_summary
Аргументы: {'topic': 'List of PlayStation Portable games', 'sentences': 2}

[HumanMessage(content='Give information about PlayStation Portable', additional_kwargs={}, response_metadata={}), ToolMessage(content="content=['PlayStation Portable', '